In [1]:
!sudo apt-get update
!sudo apt-get install libopenblas-dev -y
!pip install numpy
!pip install torch torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html
!pip install evaluate seqeval

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [2]:
! git clone https://github.com/UniversalNER/UNER_Slovak-SNK
! git clone https://github.com/UniversalNER/UNER_English-EWT

fatal: destination path 'UNER_Slovak-SNK' already exists and is not an empty directory.
fatal: destination path 'UNER_English-EWT' already exists and is not an empty directory.


In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, AutoConfig
import torch
from tqdm.auto import tqdm
from datasets import Dataset
import evaluate


In [4]:
torch.cuda.is_available()

True

In [5]:
device = "cuda" if torch.cuda.is_available() else xm.xla_device()
device

'cuda'

In [6]:
from pathlib import Path

def load_iob(file_path: str | Path) -> tuple[list[str], list[str]]:
    with open(file_path, "r", encoding="UTF-8") as f:
        lines = f.readlines()
    words = list()
    tags = list()
    for line in lines:
        if line[0] != "#" and line != "\n":
            _, word, tag, _, _ = line.strip().split("\t")
            words.append(word)
            tags.append(tag)
    return words, tags

words, tags = load_iob("UNER_Slovak-SNK/sk_snk-ud-dev.iob2")
tag_to_idx = {"O": 0, 'B-ORG': 1, 'I-ORG': 2, 'I-PER': 3, 'B-PER': 4, 'B-LOC': 5, 'I-LOC': 6, }
idx_to_tag = {i: tag for tag, i in tag_to_idx.items()}


In [7]:
ENGLISH_MODEL_NAME = "google-bert/bert-base-cased"
MULTI_MODEL_NAME = "FacebookAI/xlm-roberta-large" # "google-bert/bert-base-multilingual-cased"
LR = 2e-5
EPOCHS = 3
BATCH_SIZE = 8

In [8]:
english_tokenizer = AutoTokenizer.from_pretrained(ENGLISH_MODEL_NAME, use_fast=True)
multi_tokenizer = AutoTokenizer.from_pretrained(MULTI_MODEL_NAME, use_fast=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
english_tokenized = english_tokenizer(
    words,
    is_split_into_words=True,
    max_length=128,
    truncation="only_first", # Keep the beginning, slice the rest
    return_overflowing_tokens=True, # Returns a list of chunks
    padding="max_length",    # Ensure all chunks are exactly 128
    return_tensors="pt"      # Returns PyTorch tensors (or "tf", "np")
)

multi_tokenized = multi_tokenizer(
    words,
    is_split_into_words=True,
    max_length=128,
    truncation="only_first", # Keep the beginning, slice the rest
    return_overflowing_tokens=True, # Returns a list of chunks
    padding="max_length",    # Ensure all chunks are exactly 128
    return_tensors="pt"      # Returns PyTorch tensors (or "tf", "np")
)

In [10]:
print(english_tokenizer.convert_ids_to_tokens(english_tokenized["input_ids"][0,:15]))
print(multi_tokenizer.convert_ids_to_tokens(multi_tokenized["input_ids"][0,:15]))

['[CLS]', '"', 'N', '##ep', '##os', '##l', '##ú', '##cho', '##l', 'r', '##oz', '##ka', '##zy', ',', 'ľ']
['<s>', '▁"', '▁Nepo', 's', 'lú', 'chol', '▁roz', 'ka', 'zy', '▁', ',', '▁ľah', 'ol', '▁si', '▁na']


In [11]:
len(multi_tokenized["input_ids"])

183

In [12]:
def align_labels(tokenized, labels):
    # 2) Prepare a new "labels" list aligned to the subword tokens
    all_labels = torch.empty_like(tokenized["input_ids"], dtype=torch.int8)
    for i in range(all_labels.shape[0]):
        word_ids = tokenized.word_ids(batch_index = i)

        prev_word_id = None

        for j, word_id in enumerate(word_ids):
            if word_id is None:
                # e.g. special tokens or padding
                all_labels[i, j] = -100
            elif word_id == prev_word_id:
                # subword token of the same word => ignore
                all_labels[i, j] = -100
            else:
                # new subword, so use the label for the original token
                all_labels[i, j] = tag_to_idx[labels[word_id]]

            prev_word_id = word_id


    # 3) Attach the new "labels" to our tokenized inputs

    tokenized["labels"] = all_labels

    # 4) Return the updated dictionary
    return tokenized

# english_tokenized = align_labels(english_tokenized, tags)
multi_tokenized = align_labels(multi_tokenized, tags)

In [13]:
print(multi_tokenizer.convert_ids_to_tokens(multi_tokenized["input_ids"][0,:15]))
print(multi_tokenized["labels"][0,:15])

['<s>', '▁"', '▁Nepo', 's', 'lú', 'chol', '▁roz', 'ka', 'zy', '▁', ',', '▁ľah', 'ol', '▁si', '▁na']
tensor([-100,    0,    0, -100, -100, -100,    0, -100, -100,    0, -100,    0,
        -100,    0,    0], dtype=torch.int8)


In [14]:
def load_tokenize_align(file_path: str | Path, model: str) -> Dataset:
    words, tags = load_iob(file_path)
    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)
    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        max_length=128,
        truncation="only_first", # Keep the beginning, slice the rest
        return_overflowing_tokens=True, # Returns a list of chunks
        padding="max_length",    # Ensure all chunks are exactly 128
        return_tensors="pt"      # Returns PyTorch tensors (or "tf", "np")
    )
    tokenized = align_labels(tokenized, tags)

    return Dataset.from_dict({k: v for k, v in tokenized.items()})

In [15]:
english_config = AutoConfig.from_pretrained(ENGLISH_MODEL_NAME, num_labels=len(tag_to_idx))
english_model = AutoModelForTokenClassification.from_pretrained(
    ENGLISH_MODEL_NAME,
    config=english_config
)
# Create a data collator
english_data_collator = DataCollatorForTokenClassification(english_tokenizer)

multi_config = AutoConfig.from_pretrained(MULTI_MODEL_NAME, num_labels=len(tag_to_idx))
multi_model = AutoModelForTokenClassification.from_pretrained(
    MULTI_MODEL_NAME,
    config=multi_config
)
# Create a data collator
multi_data_collator = DataCollatorForTokenClassification(multi_tokenizer)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly init

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
from torch.utils.data import DataLoader

english_train = load_tokenize_align(r"UNER_English-EWT/en_ewt-ud-train.iob2", ENGLISH_MODEL_NAME)
english_dev = load_tokenize_align(r"UNER_English-EWT/en_ewt-ud-dev.iob2", ENGLISH_MODEL_NAME)
slovak_dev = load_tokenize_align(r"UNER_Slovak-SNK/sk_snk-ud-dev.iob2", ENGLISH_MODEL_NAME)

train_dataloader = DataLoader(english_train, shuffle=True, collate_fn=english_data_collator, batch_size=BATCH_SIZE)
dev_dataloader = DataLoader(english_dev, collate_fn=english_data_collator, batch_size=BATCH_SIZE)
slovak_dataloader = DataLoader(slovak_dev, collate_fn=english_data_collator, batch_size=BATCH_SIZE)

In [17]:
english_optimizer = torch.optim.AdamW(english_model.parameters(), lr=LR)
multi_optimizer = torch.optim.AdamW(multi_model.parameters(), lr=LR)


In [18]:
english_model.to(device)

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [19]:
multi_model.to(device)

XLMRobertaForTokenClassification(
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=1024, out_features=7, bias=True)
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
    

In [20]:
def train_model(model, dataloader, optimizer, epochs, train_whole = True):
    #if not train_whole:
    #  for param in model.bert.parameters():
    #    param.requires_grad = False
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Training Epoch {epoch+1}")
        for step, batch in pbar:
            optimizer.zero_grad()
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
        print(total_loss/len(pbar))

In [21]:
train_model(english_model, train_dataloader, english_optimizer, 20)

Training Epoch 1:   0%|          | 0/241 [00:00<?, ?it/s]

0.14268984411808341


Training Epoch 2:   0%|          | 0/241 [00:00<?, ?it/s]

0.03184264974525639


Training Epoch 3:   0%|          | 0/241 [00:00<?, ?it/s]

...

0.0019726546802812813


Training Epoch 19:   0%|          | 0/241 [00:00<?, ?it/s]

0.0008239835993942555


Training Epoch 20:   0%|          | 0/241 [00:00<?, ?it/s]

0.0006380708989100598


In [ ]:
train_model(multi_model, train_dataloader, multi_optimizer, 20)

Training Epoch 1:   0%|          | 0/241 [00:00<?, ?it/s]

0.0050068363725208715


Training Epoch 2:   0%|          | 0/241 [00:00<?, ?it/s]

0.0037762060362572254


Training Epoch 3:   0%|          | 0/241 [00:00<?, ?it/s]

...

Training Epoch 10:   0%|          | 0/241 [00:00<?, ?it/s]

0.002850257835249611


Training Epoch 11:   0%|          | 0/241 [00:00<?, ?it/s]

0.0024165797783335993


Training Epoch 12:   0%|          | 0/241 [00:00<?, ?it/s]

In [23]:
metric = evaluate.load("seqeval")

In [24]:
def get_labels(predictions, references):
    """
    Convert model outputs (logits) and references (label IDs)
    into human-readable label names, ignoring subword tokens.

    Args:
        predictions: A PyTorch tensor with shape [batch_size, seq_length],
                     containing the predicted label IDs for each token.
        references:  A PyTorch tensor with the true label IDs for each token.

    Returns:
        true_predictions, true_labels:
        - Each is a list of lists of strings.
        - Outer list = batch dimension
        - Inner list = predicted or true labels for each token in that example
        - We skip any token whose label == -100 (these are subword tokens or padding).

    Example:
        Suppose label_list = ["O", "B-PER", "I-PER"],
        predictions = [[0, 1, 2], [0, 0, 1]],
        references  = [[0, 1, 2], [0, 0, -100]]

        Then,
        true_predictions might be [["O", "B-PER", "I-PER"], ["O", "O"]]
        true_labels      might be [["O", "B-PER", "I-PER"], ["O", "O"]]
    """
    predictions = predictions.cpu().numpy()
    references = references.cpu().numpy()
    true_predictions = []
    true_labels = []
    for i, example in enumerate(references):
      true_labels.append([idx_to_tag[idx] for idx in example if idx != -100])
      true_predictions.append([idx_to_tag[idx] for j, idx in enumerate(predictions[i,:]) if references[i, j] != -100])
    return true_predictions, true_labels

In [25]:
def compute_metrics(preds, refs):
    results = metric.compute(predictions=preds, references=refs)
    return {
        "Precision": results["overall_precision"],
        "Recall": results["overall_recall"],
        "F1": results["overall_f1"],
        "Accuracy": results["overall_accuracy"],
    }

In [26]:
# After training, evaluate on the validation set
def eval_model(model, dataloader):
    model.eval()
    validation_progress_bar = tqdm(range(len(dataloader)))
    all_predictions = []
    all_labels = []
    for step, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]
        predicted_labels, true_labels = get_labels(predictions, labels)
        all_predictions.extend(predicted_labels)
        all_labels.extend(true_labels)
        validation_progress_bar.update(1)

    validation_metrics = compute_metrics(all_predictions, all_labels)
    print(validation_metrics)

In [30]:
eval_model(english_model, dev_dataloader)
eval_model(english_model, slovak_dataloader)

  0%|          | 0/32 [00:00<?, ?it/s]

{'Precision': np.float64(0.7645290581162325), 'Recall': np.float64(0.7817622950819673), 'F1': np.float64(0.773049645390071), 'Accuracy': 0.982541068169193}


  0%|          | 0/38 [00:00<?, ?it/s]

{'Precision': np.float64(0.4849246231155779), 'Recall': np.float64(0.2960122699386503), 'F1': np.float64(0.3676190476190476), 'Accuracy': 0.9452001855000773}


In [ ]:
eval_model(multi_model, dev_dataloader)
eval_model(multi_model, slovak_dataloader)